# HRP Database Setup

This notebook does the following:
1. Loading health monitoring data from Excel files
2. Setting up a local SQLite database for the data storage
3. Creating a normalized schema out of measurements, seniors, and other tables.

**Summary of Results:**
- **Seniors**: 14,170 unique seniors
- **Measurements**: 164,331,155 measurements for different types
- **Medical Info**: 8,739 seniors with disease & medication data
- **Diseases**: 161 unique diseases
- **Medications**: 1,731 unique medications
- **SOS Alerts**: 8,983 alert records
- **Storage**: Local SQLite database

In [25]:
import sys
import os
import time
import sqlite3
from pathlib import Path
import warnings

import pandas as pd

sys.path.append(os.path.abspath(".."))
from src.components.load_data import load_all_data

warnings.filterwarnings('ignore')

## Section 1: Load and Inspect Raw Excel Data

Load the new collection: twelve measurement files, medical/diseases, seniors demographics (gender, birthdate, age), and SOS alerts. Inspect structure, types, and basic quality.

In [2]:
old_data_dir = Path("../data/raw/HRP_old")
new_data_dir = Path("../data/raw/2026-02-08")

measurement_files = [
    # Old 30-day data (Nov)
    old_data_dir / "data_202512221122-01-09.xlsx",
    old_data_dir / "data_202512221231-16-23.xlsx",
    old_data_dir / "data_202512221344-24-30.xlsx",
    
    # New 60-day data (Dec - Jan)
    new_data_dir / "data-2025-12-01-07_202602091218.xlsx",
    new_data_dir / "data-2025-12-08-14_202602091252.xlsx",
    new_data_dir / "data-2025-12-15-21_202602091330.xlsx",
    new_data_dir / "data-2025-12-22-28_202602091430.xlsx",
    new_data_dir / "data-2025-12-29-2026-01-04_202602091614.xlsx",
    new_data_dir / "data-2026-01-05-11-_202602092056.xlsx",
    new_data_dir / "data-2026-01-12-18_202602100141.xlsx",
    new_data_dir / "data-2026-01-19-25_202602100226.xlsx",
    new_data_dir / "data-2026-01-26-31-.xlsx",
]

for p in measurement_files:
    assert p.exists(), f"Missing file: {p}"

In [3]:
demo_old = pd.read_excel(old_data_dir / "SeniorGenderAge_202512221409.xlsx")
demo_new = pd.read_excel(new_data_dir / "SeniorGenderAge_202602082214.xlsx")

demo_combined = pd.concat([demo_old, demo_new])
df_demo_raw = demo_combined.drop_duplicates(subset=['seniorID'], keep='last').reset_index(drop=True)

In [4]:
meds_old = pd.read_excel(old_data_dir / "Med&Diseases_202512221410.xlsx", engine="openpyxl")
meds_new = pd.read_excel(new_data_dir / "Med&Diseases_202602082215.xlsx", engine="openpyxl")

meds_combined = pd.concat([meds_old, meds_new])
df_medical = meds_combined.drop_duplicates().reset_index(drop=True)

In [5]:
sos_old = pd.read_excel(old_data_dir / "SOS_202512221411.xlsx")
sos_new = pd.read_excel(new_data_dir / "SOS_202602082216.xlsx")

sos_combined = pd.concat([sos_old, sos_new])
df_sos = sos_combined.drop_duplicates().reset_index(drop=True)

In [6]:
df_medical.shape

(9133, 3)

In [7]:
df_medical.columns

Index(['seniorID', 'diseaseNames', 'medicineNames'], dtype='object')

In [8]:
df_medical.dtypes

seniorID          int64
diseaseNames     object
medicineNames    object
dtype: object

In [9]:
df_medical.head()

,seniorID,diseaseNames,medicineNames
0,2875,"Osteoporoza,Nadciśnienie tętnicze,Arytmia serc...","Acard,Emanera,Agen,Concor,Valzek"
1,3755,"Miażdzyca,Osteoporoza","Gensulin,Beto,Furosemidum,Amlopin,Zahron,Berod..."
2,3762,"Cukrzyca,Niedoczynnośc tarczycy,Niedoczynnośc ...","Letrox,Diosminex,Valsacor,Metformax,Bibloc,Pol..."
3,3805,"Stomia,Niedosłuch,Skolioza","Pregabalin,Staveran,Neurovit"
4,4367,"Miażdżyca kończyn dolnych,Niewydolnośc układu ...","Allupol,Cipropol,Eliquis,Ezehron,Areplex"


In [10]:
df_medical.isnull().sum()

seniorID         0
diseaseNames     0
medicineNames    0
dtype: int64

In [11]:
df_demo_raw.shape

(14893, 4)

In [12]:
df_demo_raw.columns

Index(['seniorID', 'gender', 'birthDate', 'age'], dtype='object')

In [13]:
df_demo_raw.dtypes

seniorID       int64
gender        object
birthDate     object
age          float64
dtype: object

In [14]:
df_demo_raw.isnull().sum()

seniorID       0
gender         0
birthDate    339
age          339
dtype: int64

In [15]:
df_sos.shape

(14392, 3)

In [16]:
df_sos.columns

Index(['seniorID', 'alertDate', 'sosNote'], dtype='object')

In [17]:
df_sos.dtypes

seniorID              int64
alertDate    datetime64[ns]
sosNote              object
dtype: object

In [18]:
df_sos.head()

,seniorID,alertDate,sosNote
0,3205,2025-11-30 17:07:51,Alarm przypadkowy
1,3221,2025-11-25 19:38:36,Alarm przypadkowy
2,3275,2025-11-14 15:01:34,Alarm przypadkowy
3,3279,2025-11-09 11:58:31,Alarm przypadkowy
4,3283,2025-11-17 18:44:32,Alarm przypadkowy


In [19]:
df_sos.isnull().sum()

seniorID       0
alertDate      0
sosNote      127
dtype: int64

In [20]:
xls = pd.ExcelFile(measurement_files[0])
sheet_names = xls.sheet_names
print(f"Total sheets in first file: {len(sheet_names)}")
print(f"Sheet names: {sheet_names}\n")

Total sheets in first file: 26
Sheet names: ['expdata', 'expdata#1', 'expdata#2', 'expdata#3', 'expdata#4', 'expdata#5', 'expdata#6', 'expdata#7', 'expdata#8', 'expdata#9', 'expdata#10', 'expdata#11', 'expdata#12', 'expdata#13', 'expdata#14', 'expdata#15', 'expdata#16', 'expdata#17', 'expdata#18', 'expdata#19', 'expdata#20', 'expdata#21', 'expdata#22', 'expdata#23', 'expdata#24', 'expdata#25']



In [21]:
for i, sheet_name in enumerate(sheet_names[:2]):
    df = pd.read_excel(measurement_files[0], sheet_name=sheet_name, nrows=5, engine="openpyxl")
    print(f"\nSheet '{sheet_name}':")
    print(f"    Col 0 (seniorID): {df.iloc[:, 0].values[:2]}")
    print(f"    Col 1 (value): {df.iloc[:, 1].values[:2]}")
    print(f"    Col 2 (sbp): {df.iloc[:, 2].values[:2]}")
    print(f"    Col 3 (dbp): {df.iloc[:, 3].values[:2]}")
    print(f"    Col 4 (date): {df.iloc[:, 4].values[:2]}")
    print(f"    Col 5 (type): {df.iloc[:, 5].values[:2]}")


Sheet 'expdata':
    Col 0 (seniorID): [48129 48427]
    Col 1 (value): [36.3 36.6]
    Col 2 (sbp): [nan nan]
    Col 3 (dbp): [nan nan]
    Col 4 (date): ['2025-11-03T23:16:12.000000000' '2025-11-03T23:16:12.000000000']
    Col 5 (type): ['Temperature' 'Temperature']

Sheet 'expdata#1':
    Col 0 (seniorID): [48313 42183]
    Col 1 (value): [36.3 36.6]
    Col 2 (sbp): [nan nan]
    Col 3 (dbp): [nan nan]
    Col 4 (date): ['2025-11-04T07:59:41.000000000' '2025-11-04T07:59:41.000000000']
    Col 5 (type): ['Temperature' 'Temperature']


## Section 2: Store Data in SQLite Database

Use the pipeline from `src.components.load_data` to load demographics, measurements, medical info, and alerts into SQLite.

In [ ]:
# Run end-to-end load using the following pipeline
# Use streaming to avoid large memory usage and commit in batches
# Set fresh_start=True to rebuild the DB and process all sheets from scratch
load_all_data(fresh_start=True, streaming=True, batch_rows=100_000, resume=False)

INFO:src.components.load_data:Deleted existing database
INFO:src.components.database:Database initialized at c:\Users\eldar\Projects\AI-CVD\db\hrp_data.db
INFO:src.components.load_data:Loading seniors demographics from 2 files
INFO:src.components.load_data:Loaded 14893 unique senior demographic rows
INFO:src.components.load_data:Upserted 14893 seniors with demographics
INFO:src.components.load_data:Streaming measurements from c:\Users\eldar\Projects\AI-CVD\data\raw\HRP_old\data_202512221122-01-09.xlsx
INFO:src.components.load_data:  expdata: +100,000 (total 100,000)
INFO:src.components.load_data:  expdata: +100,000 (total 200,000)
INFO:src.components.load_data:  expdata: +100,000 (total 300,000)
INFO:src.components.load_data:  expdata: +100,000 (total 400,000)
INFO:src.components.load_data:  expdata: +100,000 (total 500,000)
INFO:src.components.load_data:  expdata: +100,000 (total 600,000)
INFO:src.components.load_data:  expdata: +100,000 (total 700,000)
INFO:src.components.load_data: 

In [38]:
db_path = Path("../db/hrp_data.db")
db_path.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

In [ ]:
if "conn" in locals() and conn:
    try:
        conn.close()
        print("Closed prior database connection")
    except Exception as e:
        print(f"Warning while closing prior connection: {e}")

Closed prior database connection


In [33]:
print(f"Database initialized at {db_path.as_posix()}")
print(f"Database size: {db_path.stat().st_size / 1024:.1f} KB")

Database initialized at ../db/hrp_data.db
Database size: 47233956.0 KB


In [ ]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(f"\nTables created: {[t[0] for t in tables]}")


Tables created: ['seniors', 'measurements', 'sqlite_sequence', 'medical_info', 'diseases', 'medicines', 'senior_diseases', 'senior_medicines', 'alerts', 'ingestion_state']


# Section 3: Verify/Inspect Data Integrity

In [ ]:
# Pull measurements 
df_measurements = pd.read_sql("SELECT * FROM measurements LIMIT 100000", conn)
print(df_measurements.shape)
df_measurements.head()

(100000, 7)


,id,senior_id,value,sbp,dbp,date,type
0,1,49789,36.6,None,None,2025-12-04 18:00:11,Temperature
1,2,45846,36.6,None,None,2025-12-04 18:00:11,Temperature
2,3,21927,36.5,None,None,2025-12-04 18:00:11,Temperature
3,4,38074,36.6,None,None,2025-12-04 18:00:11,Temperature
4,5,49987,36.7,None,None,2025-12-04 18:00:11,Temperature


In [ ]:
# Medical information counts
counts_med = pd.read_sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT senior_id) AS n_seniors FROM medical_info", conn)
counts_med

,n_rows,n_seniors
0,8739,8739


In [ ]:
# Alerts counts
counts_alerts = pd.read_sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT senior_id) AS n_seniors FROM alerts", conn)
counts_alerts

,n_rows,n_seniors
0,8983,4382


In [ ]:
# Preview alerts
pd.read_sql("SELECT * FROM alerts LIMIT 5", conn)

,alert_id,senior_id,alert_date,sos_note
0,1,3176,2026-01-07 19:33:57,Alarm. Nawiązano kontakt z Seniorem. Opiekun p...
1,2,3176,2025-12-10 19:30:51,Alarm przypadkowy
2,3,3176,2025-12-04 13:19:44,Alarm przypadkowy
3,4,3179,2025-12-19 13:43:28,Alarm testowy
4,5,3188,2026-01-04 06:12:46,Alarm przypadkowy


In [ ]:
# Measurement type distribution
measure_type_counts = pd.read_sql(
    "SELECT type, COUNT(*) AS cnt FROM measurements GROUP BY type ORDER BY cnt DESC",
    conn,
)
measure_type_counts

,type,cnt
0,Heartrate,37260275
1,BloodPressure,37260213
2,Temperature,37117542
3,Saturation,29911518
4,Steps,22781607


## Section 4: Query and Validate Stored Data

Execute SQL queries to retrieve data and perform basic analysis to confirm database functionality.

### Example 1: Get all measurements for a specific type

In [ ]:
query1 = """
    SELECT senior_id, value, date, type
    FROM measurements
    WHERE type = 'Heartrate'
    ORDER BY date
    LIMIT 10
"""

In [ ]:
df_example1 = pd.read_sql(query1, conn)
df_example1

,senior_id,value,date,type
0,38873,100.0,2025-12-01 00:00:10,Heartrate
1,43678,62.0,2025-12-01 00:00:10,Heartrate
2,33485,100.0,2025-12-01 00:00:10,Heartrate
3,9086,49.0,2025-12-01 00:00:10,Heartrate
4,46563,58.0,2025-12-01 00:00:10,Heartrate
5,52344,67.0,2025-12-01 00:00:10,Heartrate
6,33280,56.0,2025-12-01 00:00:10,Heartrate
7,9080,63.0,2025-12-01 00:00:10,Heartrate
8,46663,48.0,2025-12-01 00:00:10,Heartrate
9,46347,80.0,2025-12-01 00:00:10,Heartrate


### Example 2: Aggregate statistics by measurement type

In [ ]:
query2 = """
    SELECT 
        type,
        COUNT(*) as measurement_count,
        COUNT(DISTINCT senior_id) as unique_seniors,
        AVG(value) as avg_value,
        MIN(value) as min_value,
        MAX(value) as max_value,
        ROUND(AVG(value), 2) as mean
    FROM measurements
    WHERE value IS NOT NULL
    GROUP BY type
    ORDER BY measurement_count DESC
"""

In [ ]:
df_example2 = pd.read_sql(query2, conn)
df_example2

,type,measurement_count,unique_seniors,avg_value,min_value,max_value,mean
0,Heartrate,37260275,14011,73.253230,1.0,214.0,73.25
1,Temperature,37117542,13990,36.664781,36.3,127.9,36.66
2,Saturation,29911518,13960,97.049382,80.0,99.0,97.05
3,Steps,22781607,13300,2901.511171,1.0,48854.0,2901.51


### Example 3: Get measurements for a specific senior

In [ ]:
sample_senior_id = int(df_measurements.iloc[0]["senior_id"])
query3 = """
    SELECT senior_id, value, sbp, dbp, date, type
    FROM measurements
    WHERE senior_id = ?
    ORDER BY date DESC
    LIMIT 10
"""

In [ ]:
df_example3 = pd.read_sql(query3, conn, params=[sample_senior_id])
df_example3

,senior_id,value,sbp,dbp,date,type
0,49789,99.0,NaN,NaN,2026-01-13 11:25:11,Saturation
1,49789,NaN,131.0,82.0,2026-01-13 11:25:11,BloodPressure
2,49789,113.0,NaN,NaN,2026-01-13 11:25:11,Heartrate
3,49789,36.6,NaN,NaN,2026-01-13 11:25:11,Temperature
4,49789,278.0,NaN,NaN,2026-01-13 11:23:01,Steps
5,49789,99.0,NaN,NaN,2026-01-13 11:15:11,Saturation
6,49789,NaN,142.0,83.0,2026-01-13 11:15:11,BloodPressure
7,49789,122.0,NaN,NaN,2026-01-13 11:15:11,Heartrate
8,49789,36.6,NaN,NaN,2026-01-13 11:15:11,Temperature
9,49789,99.0,NaN,NaN,2026-01-13 11:05:11,Saturation


### Example 4: Query performance test

In [ ]:
start = time.time()
query4 = "SELECT * FROM measurements WHERE type = 'Heartrate' LIMIT 1000"
df_example4 = pd.read_sql(query4, conn)
elapsed = time.time() - start

In [ ]:
print(f"  Retrieved {len(df_example4)} rows in {elapsed:.4f} seconds")

  Retrieved 1000 rows in 0.0051 seconds


### Example 4: Blood Pressure Analysis

In [ ]:
query5 = """
    SELECT senior_id, sbp, dbp, date, type
    FROM measurements
    WHERE sbp IS NOT NULL AND dbp IS NOT NULL
    ORDER BY date DESC
    LIMIT 10
"""

In [ ]:
df_example5 = pd.read_sql(query5, conn)
df_example5

,senior_id,sbp,dbp,date,type
0,22282,135.0,64.0,2026-01-31 23:59:30,BloodPressure
1,13258,137.0,73.0,2026-01-31 23:59:30,BloodPressure
2,38877,125.0,70.0,2026-01-31 23:59:25,BloodPressure
3,53969,122.0,76.0,2026-01-31 23:59:23,BloodPressure
4,38965,112.0,74.0,2026-01-31 23:59:23,BloodPressure
5,53795,132.0,87.0,2026-01-31 23:59:23,BloodPressure
6,43762,146.0,81.0,2026-01-31 23:59:23,BloodPressure
7,48837,136.0,76.0,2026-01-31 23:59:23,BloodPressure
8,16457,135.0,76.0,2026-01-31 23:59:23,BloodPressure
9,53683,129.0,75.0,2026-01-31 23:59:23,BloodPressure


### Example 5: Get Blood Pressure Statistics

In [ ]:
query5_stats = """
    SELECT 
        COUNT(*) as bp_measurements,
        COUNT(DISTINCT senior_id) as seniors_with_bp,
        AVG(sbp) as avg_systolic,
        AVG(dbp) as avg_diastolic,
        MIN(sbp) as min_systolic,
        MAX(sbp) as max_systolic,
        MIN(dbp) as min_diastolic,
        MAX(dbp) as max_diastolic
    FROM measurements
    WHERE sbp IS NOT NULL AND dbp IS NOT NULL
"""

In [ ]:
df_bp_stats = pd.read_sql(query5_stats, conn)
df_bp_stats

,bp_measurements,seniors_with_bp,avg_systolic,avg_diastolic,min_systolic,max_systolic,min_diastolic,max_diastolic
0,37260213,14011,129.432691,78.566553,68.0,212.0,23.0,157.0
